# Espejos Coloniales — POC de Fast Prompting
### Racismo internalizado (autoracismo) en el Perú y su raíz colonial

**Proyecto Final — Preentrega 2**
Autor: Elvis Labaca · Curso: Prompt Engineering · Comisión N.º 96090

---

Esta *proof of concept* (POC) implementa, con técnicas de **Fast Prompting**, el generador
de piezas educativas (texto + imagen) descrito en la Preentrega 1: contenido que conecta
cinco manifestaciones actuales del autoracismo en el Perú con su raíz colonial específica
—el sistema de castas indígena o el régimen esclavista, según corresponda— dirigido a
jóvenes peruanos de 18 a 30 años.

**Qué vas a encontrar en este notebook:**
1. Configuración y control de costos.
2. Técnicas de Fast Prompting utilizadas y por qué se eligió cada una.
3. **Prompt maestro con estructura etiquetada** (XML), según feedback recibido en la
   Preentrega 1.
4. Implementación: generación de texto (modelo texto-texto) y de imagen (modelo texto-imagen).
5. Un ejemplo de **resultado esperado** para una pieza, mostrando la relación prompt → salida → objetivo.
6. **Indicadores de validación** para revisar la calidad de cada iteración.
7. Un modo **demo (`dry_run=True`)** que muestra el funcionamiento sin consumir créditos de API.
8. Un widget interactivo para generar piezas sin tocar el código.
9. Análisis de costos, conclusiones y referencias.


## 1. Instalación y configuración

Se requiere una API key de OpenAI (`OPENAI_API_KEY`) solo si se quiere ejecutar en modo
real (`dry_run=False`). En modo demo no se necesita ninguna key ni conexión a internet.

**Control de costo:** el diseño de esta POC hace **una única llamada a la API por pieza**
para el texto (en vez de una llamada por cada etapa de la Preentrega 1), usando salida
estructurada en JSON. Esto reduce en un 75% el número de consultas frente a un enfoque
ingenuo de 4 llamadas por pieza.


In [1]:
import sys, subprocess

def _ensure(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["openai", "ipywidgets", "python-dotenv"]:
    _ensure(pkg.replace("-", "_") if pkg != "python-dotenv" else "dotenv")

print("Dependencias listas.")

Dependencias listas.


In [2]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

TEXT_MODEL = "gpt-4o-mini"
IMAGE_MODEL = "dall-e-3"

API_CALLS_LOG = []

if OPENAI_API_KEY:
    print("API key detectada: el modo dry_run=False funcionará.")
else:
    print("No se detectó OPENAI_API_KEY. El notebook funciona igual en modo demo (dry_run=True).")

No se detectó OPENAI_API_KEY. El notebook funciona igual en modo demo (dry_run=True).


## 2. Técnicas de Fast Prompting utilizadas

| Técnica | Dónde se aplica | Por qué se eligió |
|---|---|---|
| **Role Prompting** | Componente `<rol>` del prompt maestro | Fijar el rol de "historiador + redactor" mejora la precisión histórica y el tono. |
| **Few-Shot Prompting** | Componente `<ejemplo>` del prompt maestro | Un ejemplo completo ancla el formato y la extensión, reduciendo la variabilidad entre piezas. |
| **Salida estructurada (JSON)** | Componente `<formato_salida>` | Permite resolver en una sola llamada lo que en la Preentrega 1 estaba dividido en varias etapas. |
| **Zero-Shot Prompting** | Función de revisión de sensibilidad | Tarea de verificación puntual que no necesita ejemplos adicionales. |
| **Prompting negativo** | Componente `<restricciones>` | Listar explícitamente qué evitar es más eficaz que solo describir lo que sí se quiere. |
| **Prompt estructurado con etiquetas (XML)** | Todo el prompt maestro (Sección 3) | Separar el prompt en bloques etiquetados (`<rol>`, `<reglas>`, `<restricciones>`, `<formato_salida>`) reduce la ambigüedad para el modelo y facilita mantener/editar cada parte por separado, en vez de un párrafo corrido. |

Respecto a la Preentrega 1: allí los prompts estaban pensados para copiarse y pegarse
manualmente. Con Fast Prompting se fusionan las etapas de investigación y redacción en un
único prompt con salida estructurada, y además —incorporando el feedback recibido— ese
prompt pasa de ser un párrafo corrido a tener una estructura etiquetada explícita.


## 3. Prompt maestro con estructura etiquetada (XML)

**Cambio incorporado tras feedback del profesor:** en la versión anterior, las
instrucciones del sistema eran un párrafo corrido. Acá se reescriben usando etiquetas
XML que separan cada componente (rol, contexto, reglas, restricciones y formato de
salida), lo que hace más explícito para el modelo —y más fácil de mantener para
nosotros— qué función cumple cada parte de la instrucción.

In [3]:
SYSTEM_PROMPT = """
<rol>
Eres historiador especializado en el Virreinato del Perú y en la historia de la
esclavización de personas afrodescendientes en América, y además redactor de contenido
educativo para redes sociales.
</rol>

<contexto>
El objetivo es producir piezas breves sobre racismo internalizado (autoracismo) en el
Perú actual, conectando cada manifestación con su raíz colonial específica: el sistema
de castas indígena o el régimen esclavista, según corresponda.
</contexto>

<reglas>
1. El texto debe ser riguroso históricamente y no acusatorio hacia el lector.
2. Nunca mezclar la raíz del sistema de castas indígena con la del régimen esclavista,
   salvo que el campo "Raíz histórica" indique explícitamente "ambos sistemas coloniales".
3. Seguir siempre la estructura de 3 partes definida en cada pedido de usuario.
</reglas>

<restricciones>
- No exceder el máximo de palabras indicado en el pedido.
- No usar tono moralizante ni culpabilizador.
- No proponer imágenes con personas reales ni estereotipos caricaturescos.
</restricciones>

<formato_salida>
Responde siempre en JSON válido, sin texto adicional fuera del JSON, con las claves:
"texto", "raiz_colonial", "prompt_imagen".
</formato_salida>
"""

FEW_SHOT_EXAMPLE_USER = """<pedido>
Manifestación: Vergüenza del apellido o el origen andino en contextos urbanos y laborales.
Raíz histórica: sistema de castas indígena.
Contexto: jerarquización colonial por origen étnico y geográfico.
</pedido>"""

FEW_SHOT_EXAMPLE_ASSISTANT = json.dumps({
    "texto": (
        "Cambiar de apellido o esconder el pueblo de origen para 'encajar' en la ciudad no es "
        "casualidad. Desde la Colonia, ser identificado como indígena o de provincia cerraba "
        "puertas: el sistema de castas premiaba el apellido español y castigaba el origen andino. "
        "¿Alguna vez sentiste que tu apellido o tu acento decían más de vos de lo que quisiste?"
    ),
    "raiz_colonial": "sistema de castas indígena",
    "prompt_imagen": (
        "Ilustracion digital conceptual, estilo editorial, paleta tierra y ocres con acentos "
        "de azul andino. Un documento de identidad estilizado cuyo nombre se desdibuja en el aire "
        "como humo, sin rostros ni texto legible, composicion centrada, formato cuadrado. Evitar "
        "estereotipos caricaturescos de rasgos indigenas."
    ),
}, ensure_ascii=False)


def build_text_prompt(tema_key: str) -> list:
    t = TEMAS[tema_key]
    user_prompt = f"""<pedido>
Manifestación: {t['manifestacion']}.
Raíz histórica: {t['raiz']}.
Contexto: {t['contexto']}.
</pedido>

<instrucciones_formato>
Genera una pieza para un post de Instagram educativo, dirigida a personas de 18 a 30 años
sin formación académica especializada.

Estructura obligatoria en 3 partes dentro de "texto":
1) situación cotidiana en tono cercano, no acusatorio;
2) origen colonial citando el mecanismo histórico concreto indicado en <pedido>;
3) una pregunta de cierre reflexiva, sin moralizar.

Máximo 80 palabras en "texto".
"prompt_imagen": una metáfora visual (no una persona real), estilo editorial, sin texto
dentro de la imagen, sin estereotipos caricaturescos.

Responde solo con este JSON, sin explicación adicional:
{{"texto": "...", "raiz_colonial": "...", "prompt_imagen": "..."}}
</instrucciones_formato>"""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": FEW_SHOT_EXAMPLE_USER},
        {"role": "assistant", "content": FEW_SHOT_EXAMPLE_ASSISTANT},
        {"role": "user", "content": user_prompt},
    ]

## 4. Base de conocimiento del proyecto

In [4]:
TEMAS = {
    "quechua": {
        "manifestacion": "Rechazo o vergüenza al hablar quechua o aimara en espacios públicos",
        "raiz": "sistema de castas indígena",
        "contexto": "mandato colonial de hispanización como condición de ascenso social",
    },
    "alisado": {
        "manifestacion": 'Alisado capilar como condición de "buena presencia" laboral',
        "raiz": "régimen esclavista",
        "contexto": "normas de moralidad pública sobre el cabello afro impuestas desde el siglo XVIII",
    },
    "mejorar_raza": {
        "manifestacion": 'La expresión "mejorar la raza" al elegir pareja',
        "raiz": "ambos sistemas coloniales",
        "contexto": "lógica de blanqueamiento común al sistema de castas y al régimen esclavista",
    },
    "apellido": {
        "manifestacion": "Vergüenza del apellido, el acento o el origen andino/amazónico",
        "raiz": "sistema de castas indígena",
        "contexto": "jerarquización colonial por origen étnico y geográfico",
    },
    "medios": {
        "manifestacion": "Sobrerrepresentación de rasgos europeos como sinónimo de belleza en medios",
        "raiz": "ambos sistemas coloniales",
        "contexto": "convergencia histórica en el canon estético peruano actual",
    },
}

list(TEMAS.keys())

['quechua', 'alisado', 'mejorar_raza', 'apellido', 'medios']

## 5. Generador de texto (modelo texto-texto)

Una sola función, una sola llamada a la API por pieza. Combina Role Prompting + Few-Shot
+ salida JSON + restricciones explícitas, usando el prompt maestro etiquetado de la
Sección 3.

In [5]:
def generate_piece(tema_key: str, dry_run: bool = True) -> dict:
    """Genera una pieza (texto + prompt de imagen) para el tema indicado.

    dry_run=True  -> no llama a la API, devuelve una salida simulada (sin costo).
    dry_run=False -> hace UNA llamada real a la API de OpenAI (Chat Completions, JSON mode).
    """
    messages = build_text_prompt(tema_key)

    if dry_run:
        t = TEMAS[tema_key]
        return {
            "texto": f"[DEMO] Pieza simulada sobre: {t['manifestacion']}. "
                     f"(raíz: {t['raiz']}). Ejecuta con dry_run=False y una API key válida "
                     f"para obtener el texto real generado por el modelo.",
            "raiz_colonial": t["raiz"],
            "prompt_imagen": f"[DEMO] Prompt de imagen simulado para: {t['manifestacion']}.",
        }

    if not OPENAI_API_KEY:
        raise RuntimeError("Falta OPENAI_API_KEY. Define la variable de entorno o usa dry_run=True.")

    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    response = client.chat.completions.create(
        model=TEXT_MODEL,
        messages=messages,
        response_format={"type": "json_object"},
        temperature=0.7,
    )
    API_CALLS_LOG.append({"tipo": "texto", "tema": tema_key, "modelo": TEXT_MODEL})

    return json.loads(response.choices[0].message.content)

## 6. Generador de imagen (modelo texto-imagen)

Reutiliza el `prompt_imagen` devuelto por el paso anterior — no se vuelve a llamar a un
modelo de texto para construir el prompt de imagen, evitando una consulta extra por
pieza.

In [6]:
def generate_image(prompt_imagen: str, dry_run: bool = True) -> str:
    """Devuelve una URL de imagen (modo real) o una ruta simulada (modo demo)."""
    if dry_run:
        return f"[DEMO] (sin llamada real) Imagen simulada para prompt: {prompt_imagen[:60]}..."

    if not OPENAI_API_KEY:
        raise RuntimeError("Falta OPENAI_API_KEY. Define la variable de entorno o usa dry_run=True.")

    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    response = client.images.generate(
        model=IMAGE_MODEL,
        prompt=prompt_imagen,
        n=1,
        size="1024x1024",
    )
    API_CALLS_LOG.append({"tipo": "imagen", "modelo": IMAGE_MODEL})

    return response.data[0].url

## 7. Función de revisión (Zero-Shot) — opcional, bajo demanda

In [7]:
def review_piece(texto: str, dry_run: bool = True) -> dict:
    if dry_run:
        return {"observaciones": "[DEMO] Sin llamada real. Revisión disponible con dry_run=False."}

    if not OPENAI_API_KEY:
        raise RuntimeError("Falta OPENAI_API_KEY. Define la variable de entorno o usa dry_run=True.")

    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    prompt = f"""Revisa el siguiente texto de una campaña educativa sobre racismo internalizado
en el Perú. Señala en JSON si hay: (1) afirmaciones históricas no verificables,
(2) tono acusatorio hacia el lector, (3) estereotipos, (4) lenguaje poco accesible.
Responde solo: {{"observaciones": "..."}}.

Texto: \"\"\"{texto}\"\"\" """

    response = client.chat.completions.create(
        model=TEXT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0.3,
    )
    API_CALLS_LOG.append({"tipo": "revision", "modelo": TEXT_MODEL})
    return json.loads(response.choices[0].message.content)

## 8. Ejemplo de resultado esperado

**Incorporado tras feedback recibido:** para dejar demostrada la relación entre
`prompt → salida → objetivo`, se muestra a continuación qué se espera que devuelva el
modelo para la pieza del tema "apellido" (el mismo usado como Few-Shot en la Sección 3),
junto con el objetivo que cumple cada parte de la salida.

| Campo de salida | Valor esperado | Objetivo que cumple |
|---|---|---|
| `texto` | "Cambiar de apellido o esconder el pueblo de origen para 'encajar' en la ciudad no es casualidad. [...] ¿Alguna vez sentiste que tu apellido o tu acento decían más de vos de lo que quisiste?" | Situación cotidiana + raíz colonial + pregunta reflexiva, en ≤80 palabras, tono no acusatorio. |
| `raiz_colonial` | "sistema de castas indígena" | Permite verificar automáticamente, por código, que no se mezcló con la raíz esclavista. |
| `prompt_imagen` | "Ilustracion digital conceptual [...] Evitar estereotipos caricaturescos de rasgos indigenas." | Prompt reutilizable en la herramienta de imagen, sin depender de una nueva llamada a un modelo de texto. |

Este ejemplo es, de hecho, el mismo que se usa como Few-Shot dentro del prompt maestro
(Sección 3): mostrarlo también acá deja explícita, fuera del prompt, la relación entre lo
que se pide y lo que se espera obtener.

## 9. Indicadores de validación

**Incorporado tras feedback recibido:** cada iteración de una pieza (real o de prueba) se
valida contra estos cuatro indicadores antes de darla por aprobada:

| Indicador | Cómo se verifica |
|---|---|
| **Precisión histórica** | La `raiz_colonial` devuelta coincide con la definida en `TEMAS` (Sección 4) para ese tema; no se mezclan ambas raíces salvo que corresponda. |
| **Adecuación al límite de palabras** | El campo `texto` no supera las 80 palabras pedidas en el prompt. |
| **Accesibilidad del lenguaje** | El texto no usa vocabulario académico o técnico; se revisa con `review_piece()` (Sección 7), que señala lenguaje poco accesible. |
| **Ausencia de estereotipos** | El `prompt_imagen` no describe personas reales ni rasgos caricaturescos; se revisa manualmente antes de generar la imagen. |

La función `validar_pieza()` de abajo automatiza el primer y segundo indicador (los
verificables por código sin otra llamada a la API); los otros dos requieren la función de
revisión (Sección 7) o una lectura manual.

In [8]:
def validar_pieza(tema_key: str, resultado: dict) -> dict:
    """Verifica automáticamente los indicadores de precisión histórica y límite de palabras."""
    raiz_esperada = TEMAS[tema_key]["raiz"]
    precision_historica_ok = resultado["raiz_colonial"] == raiz_esperada

    palabras = len(resultado["texto"].split())
    limite_palabras_ok = palabras <= 80

    return {
        "precision_historica_ok": precision_historica_ok,
        "limite_palabras_ok": limite_palabras_ok,
        "cantidad_palabras": palabras,
    }


# Ejemplo de uso en modo demo
_demo = generate_piece("apellido", dry_run=True)
validar_pieza("apellido", _demo)

{'precision_historica_ok': True,
 'limite_palabras_ok': True,
 'cantidad_palabras': 35}

## 10. Demostración en modo demo (sin costo)

In [9]:
demo_resultado = generate_piece("alisado", dry_run=True)
demo_imagen = generate_image(demo_resultado["prompt_imagen"], dry_run=True)

print("TEXTO:", demo_resultado["texto"])
print("\nRAÍZ COLONIAL:", demo_resultado["raiz_colonial"])
print("\nPROMPT DE IMAGEN:", demo_resultado["prompt_imagen"])
print("\nIMAGEN:", demo_imagen)
print("\nVALIDACIÓN:", validar_pieza("alisado", demo_resultado))

TEXTO: [DEMO] Pieza simulada sobre: Alisado capilar como condición de "buena presencia" laboral. (raíz: régimen esclavista). Ejecuta con dry_run=False y una API key válida para obtener el texto real generado por el modelo.

RAÍZ COLONIAL: régimen esclavista

PROMPT DE IMAGEN: [DEMO] Prompt de imagen simulado para: Alisado capilar como condición de "buena presencia" laboral.

IMAGEN: [DEMO] (sin llamada real) Imagen simulada para prompt: [DEMO] Prompt de imagen simulado para: Alisado capilar como ...

VALIDACIÓN: {'precision_historica_ok': True, 'limite_palabras_ok': True, 'cantidad_palabras': 32}


## 11. Widget interactivo

In [10]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

tema_dropdown = widgets.Dropdown(
    options=[(v["manifestacion"], k) for k, v in TEMAS.items()],
    description="Tema:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px"),
)

dry_run_checkbox = widgets.Checkbox(
    value=True,
    description="Modo demo (dry_run) — no consume créditos de API",
)

generar_btn = widgets.Button(description="Generar pieza", button_style="success")
salida = widgets.Output()


def on_click(_):
    with salida:
        clear_output()
        tema_key = tema_dropdown.value
        dry_run = dry_run_checkbox.value
        resultado = generate_piece(tema_key, dry_run=dry_run)
        imagen = generate_image(resultado["prompt_imagen"], dry_run=dry_run)
        validacion = validar_pieza(tema_key, resultado)
        display(Markdown(
            f"**Texto:** {resultado['texto']}\n\n"
            f"**Raíz colonial:** {resultado['raiz_colonial']}\n\n"
            f"**Prompt de imagen:** {resultado['prompt_imagen']}\n\n"
            f"**Imagen:** {imagen}\n\n"
            f"**Validación:** {validacion}"
        ))


generar_btn.on_click(on_click)
display(tema_dropdown, dry_run_checkbox, generar_btn, salida)

Dropdown(description='Tema:', layout=Layout(width='600px'), options=(('Rechazo o vergüenza al hablar quechua o…

Checkbox(value=True, description='Modo demo (dry_run) — no consume créditos de API')

Button(button_style='success', description='Generar pieza', style=ButtonStyle())

Output()

## 12. Ejecución real (opcional — consume créditos)

In [11]:
# for tema_key in TEMAS:
#     resultado = generate_piece(tema_key, dry_run=False)
#     imagen_url = generate_image(resultado["prompt_imagen"], dry_run=False)
#     print(tema_key, "->", resultado["texto"][:60], "...")
#     print("Imagen:", imagen_url)
#     print("Validación:", validar_pieza(tema_key, resultado))
#     print("-" * 80)

print("Celda desactivada por defecto para evitar consumo accidental de créditos.")

Celda desactivada por defecto para evitar consumo accidental de créditos.


## 13. Análisis de costos

| Enfoque | Llamadas por pieza | Llamadas para las 5 piezas |
|---|---|---|
| Preentrega 1 (manual, 4 etapas por pieza) | 4 | 20 |
| Esta POC (texto+imagen fusionados, revisión opcional) | 2 | 10 |


## 14. Conclusiones

Aplicar Fast Prompting sobre la propuesta de la Preentrega 1 permitió:

- **Reducir el número de consultas a la mitad** (de 4 a 2 por pieza), fusionando etapas
  mediante salida estructurada en JSON.
- **Aumentar la consistencia del formato** entre las 5 piezas gracias al ejemplo Few-Shot.
- **Estructurar el prompt maestro con etiquetas** (rol, contexto, reglas, restricciones,
  formato de salida), lo que redujo la ambigüedad y facilita mantener cada componente por
  separado, siguiendo el feedback recibido en la Preentrega 1.
- **Definir indicadores de validación explícitos** (precisión histórica, límite de
  palabras, accesibilidad, ausencia de estereotipos), en vez de una revisión informal.
- **Separar el desarrollo del costo**: todo el pipeline pudo construirse y probarse sin
  gastar créditos, dejando el gasto real acotado a una única corrida final.


## 15. Referencias

- Kogan, L., Fuchs, R. & Lay, P. (2020). *No sabía que existía tanto racismo hasta que
  entré a trabajar ahí: un estudio sobre discriminación en procesos de selección
  laboral en Lima Metropolitana.* Pontificia Universidad Católica del Perú.
- Instituto Nacional de Estadística e Informática (INEI). *Censo Nacional 2017: Perú —
  Perfil Sociodemográfico*, primera pregunta de autoidentificación étnica.
- OpenAI. *API Reference — Chat Completions.* https://platform.openai.com/docs/api-reference/chat
- OpenAI. *API Reference — Images.* https://platform.openai.com/docs/api-reference/images
